# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hrushi56/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb

con = duckdb.connect()

con.sql("""
INSTALL httpfs;
LOAD httpfs;
""")

print("DuckDB Ready")

DuckDB Ready


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [3]:
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("✅ Hugging Face secret created!")

✅ Hugging Face secret created!


In [4]:
march_df = con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5
""").df()

march_df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
print(march_df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## Unit of Analysis

For the Refresh / Content Opportunity Scoring lane, one row represents the daily search performance of a single content page. Each row contains the metrics observed for one content page on one specific date.

## Time Window

This notebook uses **March 2026** as the feature development month because it is a mid-panel month. The final month (**June 2026**) is intentionally excluded from development because it should remain an unseen outcome period to avoid data leakage during modeling.

In [6]:
grain_check = con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (content_hash_id, report_date)) AS unique_content_day_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
);
""").df()

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content_day_rows
0,9841378,9841378


## Feature Fields

The following fields will be used as model features because they are known before making a refresh decision:

- Clicks
- Impressions
- CTR (Click-Through Rate)
- Average Position
- Historical Click Trend (derived from past data only)

## Label

- Refresh Priority Score (or a future performance proxy used to rank pages for refresh)

## Context Fields

These fields identify the data but are not used directly for prediction:

- Date
- Content ID
- Client ID

## Excluded Fields

The following fields are excluded from the analysis:

- Client names
- Domains
- URLs
- Search queries
- Credentials or private identifiers

### Reason

These fields are excluded to protect client privacy, comply with the internship's public data policy, and avoid exposing confidential business information.

In [7]:
fields = con.sql("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 5;
""").df()

print("Columns:")
print(fields.columns.tolist())

fields.head()

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## Verification Queries

The following queries verify the assumptions made in the data contract.

1. Verify that one row represents one content page for one day (grain).
2. Verify the row count and date range for the selected time window (March 2026).
3. Verify data availability by filtering rows where both Search Console and Analytics data are available using `IS TRUE`.

In [8]:
# Query 1 - Row count and date span
query1 = con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
);
""").df()

print("Query 1")
display(query1)


# Query 2 - Availability check using IS TRUE
query2 = con.sql("""
SELECT
    COUNT(*) AS rows_with_both_sources
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE;
""").df()

print("Query 2")
display(query2)


# Query 3 - Data availability summary
query3 = con.sql("""
SELECT
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
GROUP BY
    gsc_data_available,
    ga4_data_available
ORDER BY rows DESC;
""").df()

print("Query 3")
display(query3)


Query 1


,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 2


,rows_with_both_sources
0,364347


Query 3


,gsc_data_available,ga4_data_available,rows
0,False,False,4690323
1,True,False,1718348
2,True,<NA>,1528366
3,False,<NA>,1490375
4,True,True,364347
5,False,True,49619


## Data Limitations

This analysis is based only on historical search performance data. It cannot directly measure content quality, user intent, external events, or Google algorithm changes. Therefore, the results should be used as decision support rather than proof of causal impact.

In [9]:
feature_frame = con.sql("""
SELECT
    report_date,
    content_hash_id,

    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)

WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE

LIMIT 20;
""").df()

feature_frame

,report_date,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,content_5c80451459c29b4a,0,5,5.400000,1,0
1,2026-03-01,content_b1f61fc81b28b2d4,0,39,5.666667,2,0
2,2026-03-01,content_e25ea7297a1dffd3,0,179,5.156425,2,0
3,2026-03-01,content_6b0149a80607dac3,0,72,7.694444,1,0
4,2026-03-01,content_62673eea26c31c17,1,3282,6.167885,1,0
5,2026-03-01,content_872342e050545a12,0,39,6.538462,1,0
6,2026-03-01,content_3c286ded8bd68120,1,88,8.431818,1,0
7,2026-03-01,content_b2108e8fe3360fa6,1,40,5.300000,1,0
8,2026-03-01,content_4c185d1c173cd53d,0,23,30.304348,1,0
9,2026-03-01,content_bd07be40ea0d5f54,0,23,5.478261,1,0


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Build a small dataset
model_df = con.sql("""
SELECT
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE
""").df()

# Simple label
model_df["high_clicks"] = (
    model_df["gsc_clicks"] >
    model_df["gsc_clicks"].median()
).astype(int)

# Honest features
X = model_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions"
    ]
]

y = model_df["high_clicks"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest Accuracy:", accuracy_score(y_test, pred))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest Accuracy: 0.6707012487992315


In [11]:
# BAD PRACTICE (Intentional Leakage)

X_leak = model_df[
    [
        "gsc_impressions",
        "gsc_avg_position",
        "ga4_sessions",
        "ga4_engaged_sessions",
        "gsc_clicks"     # <-- This directly defines the label
    ]
]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

leak_model = RandomForestClassifier(random_state=42)

leak_model.fit(X_train, y_train)

pred = leak_model.predict(X_test)

print("Leaky Accuracy:", accuracy_score(y_test, pred))

Leaky Accuracy: 1.0


## Leakage Demonstration

A deliberately leaky feature (`gsc_clicks`) was added because the target label (`high_clicks`) was created directly from this field. As expected, the model accuracy increased dramatically, showing that it had access to information that would not be available at prediction time.

After demonstrating this effect, the leaky feature was removed. The final feature set only contains variables that are available at the decision moment, ensuring an honest evaluation.

## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.